# Luach APK Builder
This notebook builds the Luach Android APK using Buildozer in Google Colab.

**Steps:**
1. Run all cells in order
2. When prompted, upload `luach_kivy.py`, `zmanim_core.py`, and `buildozer.spec`
3. The build takes ~15–25 minutes the first time (downloads Android SDK/NDK)
4. Download `Luach-debug.apk` from the last cell

In [ ]:
# Install system dependencies required by Buildozer / Android SDK
%%bash
sudo apt-get update -qq
sudo apt-get install -y \
    python3-pip build-essential git python3 python3-dev \
    libssl-dev libffi-dev \
    libsdl2-dev libsdl2-image-dev libsdl2-mixer-dev libsdl2-ttf-dev \
    libportmidi-dev libswscale-dev libavformat-dev libavcodec-dev \
    zlib1g-dev libgstreamer1.0 gstreamer1.0-plugins-base \
    autoconf libtool pkg-config \
    openjdk-17-jdk unzip wget

In [ ]:
# Install Buildozer and Cython
!pip install -q buildozer cython==0.29.36

In [ ]:
# Upload your source files:
#   luach_kivy.py
#   zmanim_core.py
#   buildozer.spec
import os
from google.colab import files

os.makedirs('/content/luach', exist_ok=True)
os.chdir('/content/luach')

uploaded = files.upload()  # pick all three files
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Verify we have all required files
required = ['luach_kivy.py', 'zmanim_core.py', 'buildozer.spec']
for f in required:
    exists = os.path.exists(f)
    print(f'  {f}: {"OK" if exists else "MISSING"}')

# Rename luach_kivy.py as main.py (Buildozer expects main.py)
if os.path.exists('luach_kivy.py') and not os.path.exists('main.py'):
    import shutil
    shutil.copy('luach_kivy.py', 'main.py')
    print('Copied luach_kivy.py -> main.py')

In [ ]:
# Patch buildozer.spec to reference main.py
with open('buildozer.spec', 'r') as f:
    spec = f.read()

spec = spec.replace(
    'source.include_patterns = luach_kivy.py,zmanim_core.py',
    'source.include_patterns = main.py,zmanim_core.py'
)
# Buildozer looks for main.py by default
if 'source.main' not in spec:
    spec = spec.replace('[app]', '[app]\nsource.main = main.py', 1)

with open('buildozer.spec', 'w') as f:
    f.write(spec)
print('buildozer.spec patched.')

In [ ]:
# Build the APK — takes 15-25 min first run (downloads ~1 GB of Android SDK/NDK)
# Subsequent builds are faster because the SDK is cached in ~/.buildozer
!buildozer -v android debug 2>&1 | tail -100

In [ ]:
# Find and download the built APK
import glob
apks = glob.glob('/content/luach/bin/*.apk')
if apks:
    apk = apks[0]
    print(f'APK built: {apk}')
    files.download(apk)
else:
    print('APK not found — check the build log above for errors')
    # Show last 50 lines of buildozer log
    import glob
    logs = glob.glob('.buildozer/android/platform/build-arm64-v8a/dists/luach/.buildozer/android/platform/build*/dist/*.log')
    if not logs:
        !tail -80 .buildozer/android/platform/build-arm64-v8a/*.log 2>/dev/null || echo 'No log found'